# Generate Precomputed Data

---

## Purpose

This notebook generates all precomputed JSON files required by the interactive artifact.

Unlike `toy_model.ipynb`, this notebook performs **no exploratory analysis**.

Its sole responsibility is to export clean JSON files consumed by

```
artifact/precomputed/
```

These files are loaded by the frontend using

```javascript
fetch("precomputed/...")
```

No scientific computation should occur inside the frontend.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
SRC = ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

from simulation import (
    build_precomputed_bundle,
)

from utils import (
    save_json,
    ensure_directory,
)

from model import (
    HebbianConfig,
)

from simulation import (
    SimulationConfig,
    generate_memory_curve,
    simulate_sequence,
    generate_metrics,
)

In [2]:
PRECOMPUTED = (
    ROOT /
    "artifact" /
    "precomputed"
)

ensure_directory(PRECOMPUTED)

print(PRECOMPUTED)

c:\Users\pankaj\Downloads\Dataforge\artifact\precomputed


In [3]:
hebb_cfg = HebbianConfig(
    n_neurons=64,
    decay=0.05,
    learning_rate=0.10,
    sparsity=0.10,
)

sim_cfg = SimulationConfig(
    n_tokens=500,
    random_seed=42,
    hidden_size=768,
    n_layers=12,
)

In [4]:
memory_curve = generate_memory_curve(
    hebb_cfg,
    sim_cfg.hidden_size,
    sim_cfg.n_layers,
)

save_json(
    memory_curve,
    PRECOMPUTED / "memory_curve.json",
)

print("memory_curve.json generated")

memory_curve.json generated


In [5]:
simulation = simulate_sequence(
    hebb_cfg,
    sim_cfg,
)

save_json(
    simulation,
    PRECOMPUTED / "simulation.json",
)

print("simulation.json generated")

simulation.json generated


In [6]:
metrics = generate_metrics(
    hebb_cfg,
    sim_cfg,
)

save_json(
    metrics,
    PRECOMPUTED / "metrics.json",
)

print("metrics.json generated")

metrics.json generated


In [7]:
bundle = build_precomputed_bundle()

save_json(
    bundle,
    PRECOMPUTED / "bundle.json",
)

print("bundle.json generated")

bundle.json generated


In [8]:
for file in PRECOMPUTED.glob("*.json"):

    print(file.name)

    print(
        round(file.stat().st_size / 1024,2),
        "KB"
    )

bundle.json
45453.44 KB
memory_curve.json
13.12 KB
metrics.json
0.22 KB
simulation.json
79707.68 KB


In [9]:
import json

with open(
    PRECOMPUTED /
    "memory_curve.json",
    "r"
) as f:

    data = json.load(f)

print(data.keys())

print()

print(data["tokens"][:10])

print()

print(data["kv_memory"][:10])

print()

print(data["hebbian_memory"][:10])

dict_keys(['tokens', 'kv_memory', 'hebbian_memory'])

[512, 1024, 1536, 2048, 2560, 3072, 3584, 4096, 4608, 5120]

[0.017578125, 0.03515625, 0.052734375, 0.0703125, 0.087890625, 0.10546875, 0.123046875, 0.140625, 0.158203125, 0.17578125]

[16.0, 16.0, 16.0, 16.0, 16.0, 16.0, 16.0, 16.0, 16.0, 16.0]


In [10]:
with open(
    PRECOMPUTED /
    "simulation.json",
    "r"
) as f:

    sim = json.load(f)

print(sim.keys())

print()

print(len(sim["frames"]))

print()

print(sim["frames"][0].keys())

dict_keys(['frames', 'norms'])

500

dict_keys(['step', 'matrix', 'norm'])


In [11]:
with open(
    PRECOMPUTED /
    "metrics.json",
    "r"
) as f:

    metrics = json.load(f)

metrics

{'tokens_processed': 500,
 'final_norm': 2.4426960945129395,
 'max_weight': 0.5808411836624146,
 'min_weight': -0.18145346641540527,
 'mean_weight': 0.003794953520224584,
 'active_synapses': 3998}

# Export Complete

The following files have been generated successfully:

```
artifact/precomputed/

memory_curve.json
simulation.json
metrics.json
bundle.json
```

These files form the contract between the Python backend and the frontend artifact.

The frontend should only **read** these JSON files and must not implement any scientific logic itself.
